# Model Training

Trains a heart disease classifier on the Heart Failure Prediction dataset (fedesoriano/Kaggle).

Each candidate model uses GridSearchCV with 10-fold stratified CV.
A soft-voting ensemble of the top 3 is also evaluated.
The best model (by held-out ROC-AUC) is refit on the full dataset and saved.

In [1]:
import json
import pathlib
import time

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier, VotingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC

In [2]:
BASE_DIR = pathlib.Path().resolve()
DATA_PATH = BASE_DIR / "data" / "heart.csv"
ARTIFACT_DIR = BASE_DIR / "app" / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

NUMERIC_FEATURES = ["Age", "RestingBP", "Cholesterol", "FastingBS", "MaxHR", "Oldpeak"]
CATEGORICAL_FEATURES = ["Sex", "ChestPainType", "RestingECG", "ExerciseAngina", "ST_Slope"]
FEATURE_ORDER = NUMERIC_FEATURES + CATEGORICAL_FEATURES
TARGET = "HeartDisease"

CV = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

In [3]:
def load_data():
    df = pd.read_csv(DATA_PATH)
    df["Cholesterol"] = df["Cholesterol"].replace(0, np.nan)
    df["RestingBP"] = df["RestingBP"].replace(0, np.nan)
    return df


def build_preprocessor():
    numeric_pipe = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ])
    categorical_pipe = Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])
    return ColumnTransformer([
        ("num", numeric_pipe, NUMERIC_FEATURES),
        ("cat", categorical_pipe, CATEGORICAL_FEATURES),
    ])


def evaluate(fitted_pipeline, X_test, y_test):
    preds = fitted_pipeline.predict(X_test)
    proba = fitted_pipeline.predict_proba(X_test)[:, 1]
    return {
        "accuracy": round(accuracy_score(y_test, preds), 4),
        "precision": round(precision_score(y_test, preds), 4),
        "recall": round(recall_score(y_test, preds), 4),
        "f1": round(f1_score(y_test, preds), 4),
        "roc_auc": round(roc_auc_score(y_test, proba), 4),
    }

## Candidate Models & Hyperparameter Grids

In [4]:
def candidate_grids():
    return {
        "logistic_regression": (
            LogisticRegression(max_iter=2000, random_state=42),
            {"model__C": [0.01, 0.1, 1, 10, 100]},
        ),
        "random_forest": (
            RandomForestClassifier(random_state=42, n_jobs=-1),
            {
                "model__n_estimators": [200, 400],
                "model__max_depth": [4, 6, 8, None],
                "model__min_samples_leaf": [1, 2, 4],
            },
        ),
        "gradient_boosting": (
            GradientBoostingClassifier(random_state=42),
            {
                "model__n_estimators": [100, 200],
                "model__max_depth": [2, 3, 4],
                "model__learning_rate": [0.01, 0.05, 0.1],
            },
        ),
        "svm": (
            SVC(kernel="rbf", probability=True, random_state=42),
            {
                "model__C": [0.1, 1, 10],
                "model__gamma": ["scale", 0.01, 0.1],
            },
        ),
    }

## Run Training

In [5]:
df = load_data()
X = df[FEATURE_ORDER]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

preprocessor = build_preprocessor()
grids = candidate_grids()

results = {}
fitted_best = {}

for name, (estimator, param_grid) in grids.items():
    t0 = time.time()
    pipe = Pipeline([("preprocess", preprocessor), ("model", estimator)])
    search = GridSearchCV(pipe, param_grid, cv=CV, scoring="roc_auc", n_jobs=-1)
    search.fit(X_train, y_train)
    elapsed = round(time.time() - t0, 1)

    holdout_metrics = evaluate(search.best_estimator_, X_test, y_test)
    holdout_metrics["cv_roc_auc_mean"] = round(search.best_score_, 4)
    holdout_metrics["best_params"] = {
        k.replace("model__", ""): v for k, v in search.best_params_.items()
    }

    results[name] = holdout_metrics
    fitted_best[name] = search.best_estimator_
    print(f"{name} ({elapsed}s): {json.dumps(holdout_metrics, indent=2)}")

print("\n--- All individual models trained ---")

logistic_regression (10.3s): {
  "accuracy": 0.8913,
  "precision": 0.8942,
  "recall": 0.9118,
  "f1": 0.9029,
  "roc_auc": 0.9346,
  "cv_roc_auc_mean": 0.9213,
  "best_params": {
    "C": 0.1
  }
}


random_forest (66.9s): {
  "accuracy": 0.8804,
  "precision": 0.8704,
  "recall": 0.9216,
  "f1": 0.8952,
  "roc_auc": 0.9295,
  "cv_roc_auc_mean": 0.9313,
  "best_params": {
    "max_depth": 6,
    "min_samples_leaf": 2,
    "n_estimators": 400
  }
}


gradient_boosting (21.1s): {
  "accuracy": 0.8859,
  "precision": 0.8857,
  "recall": 0.9118,
  "f1": 0.8986,
  "roc_auc": 0.9287,
  "cv_roc_auc_mean": 0.927,
  "best_params": {
    "learning_rate": 0.05,
    "max_depth": 2,
    "n_estimators": 100
  }
}


svm (4.1s): {
  "accuracy": 0.8804,
  "precision": 0.8774,
  "recall": 0.9118,
  "f1": 0.8942,
  "roc_auc": 0.9376,
  "cv_roc_auc_mean": 0.9211,
  "best_params": {
    "C": 10,
    "gamma": 0.01
  }
}

--- All individual models trained ---


In [6]:
# Build soft-voting ensemble from top 3 by holdout ROC-AUC
top3 = sorted(results.items(), key=lambda kv: kv[1]["roc_auc"], reverse=True)[:3]
top3_names = [name for name, _ in top3]
print(f"Building voting ensemble from: {top3_names}")

voting = VotingClassifier(
    estimators=[(name, fitted_best[name]) for name in top3_names],
    voting="soft",
)
voting.fit(X_train, y_train)
voting_metrics = evaluate(voting, X_test, y_test)
voting_metrics["cv_roc_auc_mean"] = None
voting_metrics["best_params"] = {"ensemble_of": top3_names}
results["voting_ensemble"] = voting_metrics
fitted_best["voting_ensemble"] = voting
print(f"voting_ensemble: {json.dumps(voting_metrics, indent=2)}")

Building voting ensemble from: ['svm', 'logistic_regression', 'random_forest']


voting_ensemble: {
  "accuracy": 0.875,
  "precision": 0.8624,
  "recall": 0.9216,
  "f1": 0.891,
  "roc_auc": 0.9377,
  "cv_roc_auc_mean": null,
  "best_params": {
    "ensemble_of": [
      "svm",
      "logistic_regression",
      "random_forest"
    ]
  }
}


## Select Best & Save Artifact

In [7]:
best_name = max(results, key=lambda n: results[n]["roc_auc"])
print(f"Best model: {best_name} (roc_auc={results[best_name]['roc_auc']})")

if best_name == "voting_ensemble":
    final_estimators = [(name, fitted_best[name]) for name in top3_names]
    final_model = VotingClassifier(estimators=final_estimators, voting="soft")
else:
    best_estimator, best_grid = grids[best_name]
    final_pipe = Pipeline([("preprocess", build_preprocessor()), ("model", best_estimator)])
    final_model = final_pipe.set_params(
        **{f"model__{k}": v for k, v in results[best_name]["best_params"].items()}
    )
final_model.fit(X, y)
joblib.dump(final_model, ARTIFACT_DIR / "model.joblib")
print(f"Saved model to {ARTIFACT_DIR / 'model.joblib'}")

Best model: voting_ensemble (roc_auc=0.9377)


Saved model to E:\heart-disease-app\backend\notebooks\app\artifacts\model.joblib


In [8]:
metadata = {
    "model_name": best_name,
    "feature_order": FEATURE_ORDER,
    "numeric_features": NUMERIC_FEATURES,
    "categorical_features": CATEGORICAL_FEATURES,
    "categorical_options": {
        "Sex": ["M", "F"],
        "ChestPainType": ["TA", "ATA", "NAP", "ASY"],
        "RestingECG": ["Normal", "ST", "LVH"],
        "ExerciseAngina": ["Y", "N"],
        "ST_Slope": ["Up", "Flat", "Down"],
    },
    "metrics_on_holdout": results[best_name],
    "all_candidate_metrics": results,
    "n_rows_trained_on": int(len(df)),
    "dataset": "Heart Failure Prediction (fedesoriano/Kaggle) - 918 unique patients",
    "target_meaning": {"0": "no heart disease", "1": "heart disease present"},
    "feature_descriptions": {
        "Age": "Age in years",
        "Sex": "M = male, F = female",
        "ChestPainType": "TA=typical angina, ATA=atypical angina, NAP=non-anginal pain, ASY=asymptomatic",
        "RestingBP": "Resting blood pressure (mm Hg)",
        "Cholesterol": "Serum cholesterol (mg/dl)",
        "FastingBS": "Fasting blood sugar > 120 mg/dl (1 = true, 0 = false)",
        "RestingECG": "Normal, ST=ST-T wave abnormality, LVH=left ventricular hypertrophy",
        "MaxHR": "Maximum heart rate achieved (60-202)",
        "ExerciseAngina": "Exercise-induced angina (Y = yes, N = no)",
        "Oldpeak": "ST depression induced by exercise relative to rest",
        "ST_Slope": "Slope of the peak exercise ST segment: Up, Flat, or Down",
    },
}
with open(ARTIFACT_DIR / "metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)
print(f"Saved metadata to {ARTIFACT_DIR / 'metadata.json'}")

Saved metadata to E:\heart-disease-app\backend\notebooks\app\artifacts\metadata.json
